# TinyGPT: when does the happy/sad direction appear? (E6, first 600 steps)
Re-runs the first 600 steps of the same training run (same seed, same LR schedule) with a checkpoint every 50 steps, then measures probe AUC and steering at layer 4, strength 8 on each checkpoint.
Settings: **GPU T4**, **Internet on**. Run all (~20 min). Download `/kaggle/working/tinygpt_early_results.zip`.

In [ ]:
import os, subprocess
REPO = "https://github.com/arcyadav/tiny-gpt-and-qwen-steering-study.git"
os.chdir("/kaggle/working")
if not os.path.exists("project"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "project"], check=True)
os.chdir("/kaggle/working/project")
print(os.listdir())

In [ ]:
!pip install -q -U transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data: 500k TinyStories (~114M tokens) + our BPE tokenizer

In [ ]:
!python -u -m tinygpt.data --out data --limit 500000

## 2. Train the first 600 steps, checkpoint every 50

In [ ]:
!python -u train.py --data data --out results/tinygpt_early --stop-step 600 --ckpt-every 50 --eval-every 50 2>&1 | grep -E 'params|val loss|done'

## 3. E6 on the 13 checkpoints (layer 4, strength 8 = best setting from the full run)

In [ ]:
!python -u -m steer.experiments --model tinygpt --run-dir results/tinygpt_early --tokenizer data/tokenizer.json --exp e6 --layer 4 --alpha 8 --out results/tinygpt_early_steer 2>&1 | grep -v 'Loading weights'

## 4. Download

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/export_early/results/tinygpt_early", exist_ok=True)
shutil.copytree("results/tinygpt_early_steer", "/kaggle/working/export_early/results/tinygpt_early_steer", dirs_exist_ok=True)
for f in ["summary.json", "log.json"]:
    shutil.copy(f"results/tinygpt_early/{f}", "/kaggle/working/export_early/results/tinygpt_early/")
shutil.make_archive("/kaggle/working/tinygpt_early_results", "zip", "/kaggle/working/export_early")
print("download /kaggle/working/tinygpt_early_results.zip from the Output panel")